In [1]:
!pip install transformers torch scikit-learn pandas

In [2]:
import pandas as pd
import torch
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

C:\Users\KIIT0001\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\KIIT0001\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
df = pd.read_csv("emotion_dataset.csv")

df = df.sample(8000, random_state=42)

df.head()

,text,emotion_vector,stress
2388,Help with what?,"[0, 0, 0, 0, 1]",0
4304,It more means nuxia needs help.,"[0, 0, 0, 0, 1]",0
8994,"Sony, for similar design on generic game contr...","[0, 0, 0, 0, 1]",0
14613,This is being said about every fucking update ...,"[0, 0, 1, 0, 0]",2
8087,There's a guy that died after he let a horse d...,"[0, 1, 0, 0, 0]",1


In [4]:
emotions = ["joy", "sadness", "anger", "fear", "neutral"]

def get_emotion_label(vector):
    if isinstance(vector, str):
        vector = eval(vector)
    for i, val in enumerate(vector):
        if val == 1:
            return emotions[i]
    return "neutral"

df["emotion"] = df["emotion_vector"].apply(get_emotion_label)

In [5]:
def combine_label(row):
    return f"{row['emotion']}_{row['stress']}"

df["final_label"] = df.apply(combine_label, axis=1)

In [6]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["final_label"])

print("Total classes:", len(le.classes_))

Total classes: 8


In [7]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

In [9]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [10]:
train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)

In [11]:
num_labels = len(df["label"].unique())

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
training_args = TrainingArguments(
    output_dir="./emotion_results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    max_steps=500,   
    weight_decay=0.01,
    logging_steps=10
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Step,Training Loss
10,1.919756
20,1.534265
30,1.104141
40,1.069463
50,1.101549
60,1.294305
70,0.888184
80,1.012763
90,1.041568
100,0.862471


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.8732720446586609, metrics={'train_runtime': 928.9644, 'train_samples_per_second': 2.153, 'train_steps_per_second': 0.538, 'total_flos': 59513206560000.0, 'train_loss': 0.8732720446586609, 'epoch': 0.3125})

In [14]:
print("Training finished")

Training finished


In [15]:
trainer.evaluate()

Training Loss,Validation Loss,Step
0.825486,0.743865,500


{'eval_loss': 0.7438652515411377}

In [16]:
model.save_pretrained("emotion_model")
tokenizer.save_pretrained("emotion_model")

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
from transformers import pipeline

classifier = pipeline("text-classification", model="emotion_model", tokenizer="emotion_model")

with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

def predict(text):
    result = classifier(text)[0]
    label_index = int(result['label'].split("_")[-1])
    decoded = le.inverse_transform([label_index])[0]

    emotion, stress = decoded.split("_")

    stress_map = {
        "0": "Low",
        "1": "Medium",
        "2": "High"
    }

    return {
        "emotion": emotion.capitalize(),
        "stress": stress_map.get(stress, stress),
        "confidence": round(result['score'], 3)
    }

print(predict("I feel very anxious today"))
print(predict("I am so happy right now"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'emotion': 'Sadness', 'stress': 'Medium', 'confidence': 0.282}
{'emotion': 'Joy', 'stress': 'Low', 'confidence': 0.452}
